# LLM Text Annotation: Judging and Improving Output Quality with Local Fine-Tuning

Labelling text by hand is slow, so researchers ask models to do it. This notebook takes one labelling task, measures how well a small open model does it, fine-tunes the model to do it better, and then checks whether any of that survives a move to a different set of comments. Every model run was done once on a graphics card and saved, so the notebook reads the saved answers and gives the same numbers every time.

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, ".")  

import numpy as np
import pandas as pd

from src import config as cfg, data, nyt, viz
from src.annotate import annotate, cached_records
from src.prompts import RUBRIC

pd.set_option("display.max_colwidth", 90)
viz.style()

for k, v in cfg.summary().items():
    print(f"{k:>14}: {v}")

         model: unsloth/Qwen3-1.7B-unsloth-bnb-4bit
          seed: 20260810
      eval set: 400 comments (200 per class)
     probe set: 300 comments
 saved answers: 3 model runs in cache/


## 1. Labelling comments at scale

The SFU Opinion and Comments Corpus collects reader comments from *The Globe and Mail*. Suppose we want to know whether comment sections get less constructive during election campaigns. Before we can answer that, every comment needs a label, and there are far too many to read.

In [2]:
articles = pd.read_csv(cfg.ARTICLES_CSV, usecols=["article_id", "title", "published_date", "ncomments"])
gold = data.load_gold()

print(f"articles in the corpus : {len(articles):,}")
print(f"comments on them       : {articles.ncomments.sum():,.0f}")
print(f"hand-labelled by humans: {len(gold):,}")

articles in the corpus : 10,339
comments on them       : 663,173
hand-labelled by humans: 1,043


Someone labelled 1,043 of them by hand. That set is our answer key, and the rest of the notebook is about whether a model can extend it to the other 662,000.

A comment counts as **constructive** if it tries to add something: a specific point, evidence, a personal experience, a proposed solution. A comment is **not constructive** if it is only an insult, a one-line dismissal, or sarcasm with nothing behind it. Here is one of each.

In [3]:
for label in ("yes", "no"):
    row = gold[gold.is_constructive == label].iloc[1]
    print(f"[constructive: {label}]  {row.comment_text[:290]}\n")

[constructive: yes]  Everyone is still missing the point of what the Apple Watch is:It's fashion. It is by definition of no utility. Watches have been more fashion than function for as long as they've been worn. Anyone remember paying $50 for a Swatch that cost $2 to make? The Apple Watch actually offers quite

[constructive: no]  You may be using a blackberry. I'm still using a gooseberry.



In [4]:
print(gold.is_constructive.value_counts().to_string())
print(f"\ndrawn from {gold.article_id.nunique()} articles")

eval_set, probe_set = data.gold_splits()
print(f"\nevaluation set: {len(eval_set)} comments, half of them constructive")
print(f"probe set     : {len(probe_set)} comments, kept separate")

is_constructive
yes    554
no     489

drawn from 13 articles

evaluation set: 400 comments, half of them constructive
probe set     : 300 comments, kept separate


The already labelled articles (which act as our answer key) are nearly balanced, and it comes from only a handful of articles, so it is a narrow slice of the corpus. We split it once, now. The evaluation set is scored once per method and never decides anything: no setting and no model is chosen by looking at it. The probe set is used later to watch the model during training.

## 2. Asking a model to annotate

The model is **Qwen3-1.7B**, an open-weights model small enough to run on a home graphics card. It is stored at 4-bit precision, which squeezes each weight into a quarter of the usual space and shrinks the model to about 1.3 GB.

Zero-shot means we describe the task and ask, with no examples and no training.

In [5]:
print(RUBRIC)

You are annotating reader comments from a Canadian news website.

A comment is CONSTRUCTIVE if it tries to add something to the conversation: it makes a specific point, gives evidence or a personal experience, offers a solution, or engages with the article's argument.

A comment is NOT CONSTRUCTIVE if it is only an insult, a one-line dismissal, sarcasm with no substance, off-topic ranting, or an unsupported assertion.

Comment:
"""{comment}"""

Reply in exactly this format and nothing else:
LABEL: yes
REASON: <one short sentence>


Each comment went through that prompt once, and the answer was saved under a key made from the model, the prompt and the comment. `annotate` reads those answers back.

In [6]:
zero_shot = annotate(eval_set.comment_counter, RUBRIC)

pd.DataFrame({"comment": eval_set.comment_text.str[:70],
              "human": eval_set.is_constructive,
              "model": zero_shot}).head(8)

,comment,human,model
0,Plenty. Ever been to Vancouver? There are condo boards that refuse to,no,no
1,Great piece! Thanks.,no,no
2,Tail wagging the dog,no,no
3,"Apparently, trying not to offend and be politically correct just doesn",no,no
4,The Belgian jihadis come mainly from the Rif mountains in northern Mor,no,no
5,Just keep whining - next thing that will happen is a ban on foreign bu,no,no
6,Why does the Globe and Mail even publish such a simplistic and accusat,no,no
7,"ROTFLMAO - hellloooo , no insurance, tax fraud,,, great starts, not to",no,no


It is worth looking at what the model actually wrote, not just the label we parsed out of it. Asking for a fixed format is what makes the output usable as data.

In [7]:
records = cached_records(RUBRIC)
for cid in eval_set.comment_counter.head(3):
    print(records[cid]["raw"], "\n" + "-" * 60)

LABEL: no
REASON: The comment is not constructive because it is an insult and lacks specific evidence or a personal experience. 
------------------------------------------------------------
LABEL: no
REASON: The comment is a one-line dismissal and does not add anything to the conversation. 
------------------------------------------------------------
LABEL: no
REASON: The comment is a sarcastic and unsupported assertion. 
------------------------------------------------------------


## 3. Judging the labels

Accuracy is the share the model got right. **Cohen's kappa** (add formula, definitions and how to understand later) subtracts the agreement you would expect from chance, so 0 means no better than guessing and 1 means perfect.

In [8]:
zs = data.metrics(eval_set.is_constructive, zero_shot)
pd.Series(zs).round(3).to_frame("zero-shot")

,zero-shot
accuracy,0.730
f1,0.649
cohen_kappa,0.460
predicted_yes,0.270
unparsed,0.000


Kappa around 0.46 counts as poor agreement. The confusion matrix shows what shape the errors take.

In [9]:
data.confusion(eval_set.is_constructive, zero_shot)

,model said constructive,model said not constructive,share the model agreed
humans said constructive,100,100,0.50
humans said not constructive,8,192,0.96


Look at the last column. The model agrees with the humans on 96% of the comments they called not constructive, and on half of the ones they called constructive. It is not randomly unreliable. It has one habit: saying "not constructive" too often. That habit causes almost all of its errors.

Before deciding whether 0.46 is bad, we need something to compare it against. About a fifth of the answer key was labelled a second time by an expert, and the crowd and the expert do not always agree either.

In [10]:
from sklearn.metrics import cohen_kappa_score

has_expert = gold[gold.expert_is_constructive.notna()]
crowd = (has_expert.is_constructive.str.lower() == "yes").astype(int)
expert = (has_expert.expert_is_constructive.str.lower() == "yes").astype(int)

print(f"{len(has_expert)} comments were labelled twice")
print(f"crowd and expert agree on {(crowd == expert).mean():.1%} of them")
print(f"as kappa, that is {cohen_kappa_score(crowd, expert):.3f}")

214 comments were labelled twice
crowd and expert agree on 78.5% of them
as kappa, that is 0.583


## 4. Fine-tuning with QLoRA

Prompting can only rearrange what the model already does. Fine-tuning changes the model itself, by showing it labelled examples and adjusting its weights when it gets them wrong.

Doing that the ordinary way means updating all 1.7 billion weights, which needs far more memory than we have. **QLoRA** avoids it: the original weights stay frozen at 4 bits, and we attach small trainable matrices onto the frozen model and train only those. Under 2% of the model is trainable, and what we save at the end is one small file of adapter weights.

The training comments come from C3, a larger set of 12,000 comments labelled the same way.

In [11]:
LORA = dict(
    r=16,                      # size of the bolted-on matrices
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",     # attention
                    "gate_proj", "up_proj", "down_proj"],       # and the MLP
)

TRAIN_ARGS = dict(
    per_device_train_batch_size=2,     # 8 GB of VRAM does not allow more
    gradient_accumulation_steps=8,     # so accumulate to an effective batch of 16
    max_steps=cfg.TRAIN_STEPS,
    learning_rate=2e-4,
    fp16=True, bf16=False,             # Turing GPUs have no native bf16
    gradient_checkpointing=True,       # trade compute for memory
)

Each training example is the same prompt we have been using, paired with the human label. The loss is computed **only on the label**, so the model is never rewarded for reproducing the prompt back to us.

In [12]:
train_log = json.loads((cfg.ARTIFACTS / "train_log.json").read_text())

print(train_log["trainable"])
print(f"{train_log['n_train']} training examples, {cfg.TRAIN_STEPS} steps")
print(f"{train_log['train_seconds'] / 60:.1f} minutes at {train_log['seconds_per_step']} s/step")
print(f"peak VRAM {train_log['peak_vram_mib']} MiB, final loss {train_log['final_loss']}")

17,432,576 trainable of 1,052,238,848 (1.66%)
2000 training examples, 200 steps
23.9 minutes at 7.18 s/step
peak VRAM 4899 MiB, final loss 0.2068


## 5. When did training stop helping?

A falling loss tells us the model fitted the training data. It does not tell us whether the labels got better. So during training we paused every 20 steps and pushed the same 300 probe comments through the model, recording the answer it would have given. Those comments were never trained on.

In [13]:
checkpoints = dict(np.load(cfg.ARTIFACTS / "probe_checkpoints.npz"))
curve = viz.learning_curve(checkpoints, cfg.MEDIA / "learning.png")

pd.DataFrame({"step": curve["steps"],
              "agreement (kappa)": curve["kappa"].round(3),
              "probe accuracy": curve["probe_accuracy"].round(3)})

,step,agreement (kappa),probe accuracy
0,0,0.373,0.837
1,20,0.620,0.853
2,40,0.520,0.850
3,60,0.667,0.857
4,80,0.673,0.860
5,100,0.660,0.860
6,120,0.667,0.860
7,140,0.680,0.863
8,160,0.687,0.860
9,180,0.693,0.863


![Left: agreement with the human labels on 300 held-out comments at each training checkpoint, with a shaded range. Right: how accurately a straight line can recover the human label from the model's internal state, which stays flat.](media/learning.png)

Almost all of the gain arrives in the first 60 steps. Agreement climbs from 0.37 to 0.67 and then stops. We could have trained for a third as long and finished with the same annotator.

The right panel asks something else. If we take the model's internal state and try to recover the human label from it with a straight line, how well does that work? Barely better at the end than at the start, 0.84 to 0.86. The two groups were already separable inside the model before we trained anything. Training changed the model's answers. It did not change what the model could tell apart.

## 6. Does it label better?

The same 400 comments, the same prompt, the model we just trained.

In [14]:
fine_tuned = annotate(eval_set.comment_counter, RUBRIC, adapter=cfg.ADAPTER)
truth = eval_set.is_constructive

wrong_before, wrong_after = viz.comparison_grid(truth, zero_shot, fine_tuned,
                                                cfg.MEDIA / "before_after.png")
fixed = sum(z != t and f == t for t, z, f in zip(truth, zero_shot, fine_tuned))
broke = sum(z == t and f != t for t, z, f in zip(truth, zero_shot, fine_tuned))

print(f"zero-shot got {wrong_before} of {len(truth)} wrong; fine-tuned got {wrong_after} wrong")
print(f"fine-tuning fixed {fixed} comments and broke {broke}")

zero-shot got 108 of 400 wrong; fine-tuned got 47 wrong
fine-tuning fixed 72 comments and broke 11


![Two grids of 400 squares, one per evaluation comment, with the squares the model got wrong picked out. Comments humans called constructive sit above the dividing line. The zero-shot grid is heavily marked across its top half; the fine-tuned grid is mostly clear.](media/before_after.png)

Each square is one comment, and the ones above the line are those humans called constructive. After fine-tuning the top half mostly clears which was our orignal big issue.

In [15]:
pd.DataFrame({"zero-shot": data.metrics(truth, zero_shot),
              "fine-tuned": data.metrics(truth, fine_tuned)}).round(3)

,zero-shot,fine-tuned
accuracy,0.730,0.882
f1,0.649,0.875
cohen_kappa,0.460,0.765
predicted_yes,0.270,0.438
unparsed,0.000,0.000


Fine-tuning works. Kappa goes from 0.46 to 0.77, and the model now calls 44% of comments constructive against a true rate of 50%, so the habit is gone.

The crowd and the expert agreed with each other at kappa 0.58, so the model now agrees with the crowd's labels more closely than the expert does. That does not make it better than the expert. It has just learned off of the regular crowd.

## 7. Does it work on different data?

A label is only useful if it means the same thing on data the model has not seen. So we move to *The New York Times*, where editors mark a small number of reader comments as picks.

We turn that into a game. Take two comments from the same article, one an editor's pick and one not, and ask which is which. Guessing gets 50%. The two comments in a pair are also matched on length, within 20% of each other, so a model cannot win this game by simply preferring longer comments.

In [16]:
scored = nyt.load_scores()
print(f"{len(scored) // 2:,} pairs from {scored.article_id.nunique()} articles, "
      f"{len(scored):,} comments\n")

# Two real pairs. In each one, which comment did an editor pick?
answers = []
for n, pair_id in enumerate(scored.pair_id.unique()[:2], start=1):
    both = scored[scored.pair_id == pair_id].sample(frac=1, random_state=cfg.SEED + n)
    answers.append("AB"[list(both.is_pick).index("yes")])
    print(f"--- pair {n} ---")
    for letter, (_, row) in zip("AB", both.iterrows()):
        print(f"  {letter}: {row.comment_text[:260]}\n")

print("editor picked:", ", ".join(f"pair {i} = {a}" for i, a in enumerate(answers, 1)))

1,001 pairs from 366 articles, 2,002 comments

--- pair 1 ---
  A: For decades, Trump viewed the law with contempt. It certainly never constrained him from stiffing contractors or committing fraud. And Trump knew how to use his money and his lawyers to bully and threaten anyone who stood in his way.
It always worked for him.


  B: Oh I see, so the FBI is suddenly biased in favor of Democrats? Is that why Comey (a registered Republican) delivered a public rebuke of Hillary during the campaign, breaking with FBI norms and guidelines, rather than simply state that they could find no basis 

--- pair 2 ---
  A: Nothing so represents the Trump administration better than Mick Mulvaney's performance this afternoon, fresh from Ash Wednesday service with a cross smudged on his forehead, telling congress today that Trump's military parade will cost between 10M-30M, meanwhi

  B: If he's got to have his parade, then let him have it. Let him pay for it with food boxes and let him watch it alone. 

Here is how the models did across all 1,001 pairs.

In [17]:
pd.DataFrame({"scores the editor's pick higher":
              {"always guessing": 0.5,
               "zero-shot": nyt.game_score(scored, "zero-shot"),
               "fine-tuned": nyt.game_score(scored, "fine-tuned")}}).round(3)

,scores the editor's pick higher
always guessing,0.500
zero-shot,0.551
fine-tuned,0.595


Fine-tuning helped here too. The model went from 55% to 60%, and a gap that size over 1,001 pairs is significant (maybe add some CI or p-value here depending on how technical audience is). It is a much smaller gain than the one on the Globe and Mail comments, but it is there: something the model learned about Canadian news comments transferred to a different newspaper in the United States.

## 8. Now ask it to label them

Winning the game only requires ranking one comment above another. Actual annotation requires a straight answer on each comment by itself. So we ask the fine-tuned model the same question it was trained on, one comment at a time.

In [18]:
rows = {}
for name in ("zero-shot", "fine-tuned"):
    answers = nyt.labels(scored, name)
    rows[name] = {"calls it a pick": nyt.pick_rate(scored, name),
                  "agreement (kappa)": data.metrics(scored.is_pick, answers)["cohen_kappa"]}
pd.DataFrame(rows).round(3)

,zero-shot,fine-tuned
calls it a pick,0.576,0.850
agreement (kappa),0.076,0.008


The fine-tuned model calls 85% of the comments an editor's pick, and its agreement collapses to 0.01. It is useless as an annotator here, on exactly the comments where it just won the game. Two things went wrong:

**First, it leaned on a clue that does not carry over across data.** In the Globe and Mail set, constructive comments were simply longer.

In [19]:
data.length_by_class()

,comments,median length,longest tenth
humans said,,,
constructive,554,433 characters,over 1026 characters
not constructive,489,111 characters,over 243 characters


A constructive comment in the answer key is four times the length of a non-constructive one. A model can score well on that corpus by partly learning "long means constructive." At the Times, the two comments in a pair were matched for length on purpose.

So we rebuilt the training sample, keeping the same number of comments in each length band for both classes, and trained a second model on it. Everything else about the recipe was identical.

**Second, its bar for saying yes was in the wrong place.** The model learned to say yes where about half of comments qualify. At the Times almost every comment clears that bar. Moving the bar needs no retraining, just a separate set of 100 pairs, kept aside for this and scored nowhere else.

In [20]:
socc = {"zero-shot": zero_shot, "fine-tuned": fine_tuned,
        "length-balanced": annotate(eval_set.comment_counter, RUBRIC,
                                    adapter=cfg.ADAPTER_BALANCED)}

pd.DataFrame({name: {
    "Globe and Mail: agreement (kappa)": data.metrics(truth, answers)["cohen_kappa"],
    "New York Times: wins the game": nyt.game_score(scored, name),
    "New York Times: agreement, bar moved": data.metrics(
        scored.is_pick, nyt.labels(scored, name, cfg.NYT_BAR[name]))["cohen_kappa"],
} for name, answers in socc.items()}).round(3)

,zero-shot,fine-tuned,length-balanced
Globe and Mail: agreement (kappa),0.460,0.765,0.555
New York Times: wins the game,0.551,0.595,0.618
"New York Times: agreement, bar moved",0.075,0.055,0.161


Read the two New York Times rows across, then look back at the first row. The length-balanced model is the best of the three at the Times, at 62% on the game and 0.16 agreement once its bar is moved. It is also clearly worse on the Globe and Mail comments, 0.56 against 0.77.

The first model was a crammer: it memorised what answers looked like on the practice exam, including a shortcut the exam happened to allow (comment lenght), and it aced that exam. The second model was denied the shortcut, scored lower on the practice exam, and did better on a different slightly different exam (NYT).

Here is one pair where the difference shows.

In [21]:
ex = nyt.example_pair(scored, missed_by="fine-tuned", separated_by="length-balanced")

print("EDITOR'S PICK :", ex["pick"][:280], "\n")
print("NOT PICKED    :", ex["other"][:280], "\n")
print(f"fine-tuned      scored them {ex['fine-tuned'][0]:.2f} and {ex['fine-tuned'][1]:.2f}"
      "  -> calls both a pick")
print(f"length-balanced scored them {ex['length-balanced'][0]:.2f} and "
      f"{ex['length-balanced'][1]:.2f}  -> tells them apart")

EDITOR'S PICK : If the president was truly unconcerned about the Mueller investigation, he’d simply put it behind him and not talk or tweet about it. Running the country is what he should be doing and not second-guessing the Special Prosecutor.

Mueller’s next move is anybody’s guess. He could b 

NOT PICKED    : How can there be no charges of collusion for Trump? His son according to Steve Bannons book did a treasonous act meeting with the Russians for bad info on Hillary . None was found. Then Trump in his own loose lips on public tv during a rally asked Russia for help in finding 30,00 

fine-tuned      scored them 1.00 and 1.00  -> calls both a pick
length-balanced scored them 0.84 and 0.63  -> tells them apart


### Data and licences

- **SOCC** and its constructiveness subset: Kolhatkar, V., H. Wu, L. Cavasso, E. Francis, K. Shukla and M. Taboada (2020). The SFU Opinion and Comments Corpus: A corpus for the analysis of online news comments. *Corpus Pragmatics* 4(2), 155-190. https://doi.org/10.1007/s41701-019-00065-w Licensed CC BY-NC-SA 4.0.
- **C3**: Kolhatkar, V., N. Thain, J. Sorensen, L. Dixon and M. Taboada (2020). *C3: The Constructive Comments Corpus.* Jigsaw and Simon Fraser University. DOI: 10.25314/ea49062a-5cf6-4403-9918-539e15fd7b52 Licensed CC BY-NC 4.0.
- **New York Times comments**: Kesarwani, A. *New York Times Comments.* Kaggle. https://www.kaggle.com/datasets/aashita/nyt-comments Licensed CC BY-NC-SA 4.0.
- **Model**: Qwen3-1.7B, Apache 2.0, 4-bit build by Unsloth.

### References

- Fang, Q., J. Garcia Bernardo and E-J. van Kesteren (2026). *A Methodological Guide on Using Large Language Models for Text Annotation in the Social Sciences and Humanities with Python and R.* https://arxiv.org/abs/2604.09638
- Dettmers, T., Pagnoni, A., Holtzman, A., & Zettlemoyer, L. (2023). *QLoRA: Efficient finetuning of quantized LLMs.* https://arxiv.org/abs/2305.14314
- Hu, E., et al. (2021). *LoRA: Low-rank adaptation of large language models.* https://arxiv.org/abs/2106.09685
- Kolhatkar, V. and M. Taboada (2017). Using New York Times Picks to identify constructive comments. *Proceedings of the Workshop Natural Language Processing Meets Journalism, EMNLP.* https://www.aclweb.org/anthology/W17-4218/
- Krippendorff, K. (2018). *Content Analysis: An Introduction to Its Methodology.* SAGE. On treating annotation as measurement.

### NEXT STEPS:

- Proper notebook structure
- Visualizations for the improvement and students can interact with it 
- Technical definitions zero-shot etc
- Proper explanations